# PFT Flow Execution Monitor

Monitors a running or completed `test_pft_flow` execution.

**Go/no-go criterion**: `pft_test_validation_log` shows **1000/1000** for all 5 data types across all 10 facilities.

Configure `EXECUTION_ID` below (leave empty to use the latest execution).

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
# Set to a specific execution ID, or leave empty to auto-detect the latest one.
EXECUTION_ID = ""

PG_HOST = "localhost"
PG_PORT = 54321
PG_DBNAME = "noetl"
PG_USER = "noetl"
PG_PASSWORD = "noetl"
# ─────────────────────────────────────────────────────────────────────────────

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    dbname=PG_DBNAME, user=PG_USER, password=PG_PASSWORD
)
conn.autocommit = True
print(f"Connected to PostgreSQL {PG_HOST}:{PG_PORT}/{PG_DBNAME}")

In [ ]:
def q(sql, params=None):
    """Run a query and return a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


# Auto-detect latest execution that ran test_pft_flow
if not EXECUTION_ID:
    df_exec = q("""
        SELECT execution_id, playbook_name, status, created_at, updated_at
        FROM noetl.execution_log
        WHERE playbook_name ILIKE '%pft_flow%'
        ORDER BY created_at DESC
        LIMIT 5
    """)
    display(df_exec)
    if not df_exec.empty:
        EXECUTION_ID = str(df_exec.iloc[0]["execution_id"])
        print(f"\nAuto-selected execution: {EXECUTION_ID}")
    else:
        print("No pft_flow executions found.")
else:
    print(f"Using configured execution: {EXECUTION_ID}")

## Execution Overview

In [ ]:
df_overview = q("""
    SELECT
        execution_id,
        playbook_name,
        status,
        created_at,
        updated_at,
        updated_at - created_at AS duration
    FROM noetl.execution_log
    WHERE execution_id = %s
""", (EXECUTION_ID,))
display(df_overview)

## Work Queue — Progress by Facility & Data Type

Shows `pending / claimed / done / error` counts for each (facility, data_type) pair.  
All cells should show **done=1000** when complete.

In [ ]:
df_queue = q("""
    SELECT
        facility_mapping_id,
        data_type,
        COUNT(*) FILTER (WHERE status = 'pending')  AS pending,
        COUNT(*) FILTER (WHERE status = 'claimed')  AS claimed,
        COUNT(*) FILTER (WHERE status = 'done')     AS done,
        COUNT(*) FILTER (WHERE status = 'error')    AS error,
        COUNT(*)                                     AS total
    FROM public.pft_test_patient_work_queue
    WHERE execution_id = %s
    GROUP BY facility_mapping_id, data_type
    ORDER BY facility_mapping_id, data_type
""", (EXECUTION_ID,))

if df_queue.empty:
    print("No queue rows found for this execution.")
else:
    # Pivot: facilities as rows, data_type as columns (done count)
    pivot = df_queue.pivot_table(
        index="facility_mapping_id",
        columns="data_type",
        values="done",
        aggfunc="sum",
        fill_value=0
    )
    print("Done counts per facility × data_type (target: 1000 each)")
    display(pivot)
    print()
    print("Full detail:")
    display(df_queue)

## Validation Log

Written by `validate_facility_results` after each facility completes.  
All values should equal **1000** when the run passes.

In [ ]:
df_val = q("""
    SELECT
        facility_mapping_id,
        assessments_done,
        conditions_done,
        medications_done,
        vital_signs_done,
        demographics_done,
        assessments_queue_done,
        total_expected,
        logged_at
    FROM public.pft_test_validation_log
    WHERE execution_id = %s
    ORDER BY facility_mapping_id
""", (EXECUTION_ID,))

if df_val.empty:
    print("No validation log entries yet.")
else:
    # Highlight any cell that is not 1000
    data_cols = ["assessments_done", "conditions_done", "medications_done",
                 "vital_signs_done", "demographics_done", "assessments_queue_done"]

    def highlight(val):
        if pd.isna(val):
            return "background-color: #fffbe6"
        return "background-color: #d4edda" if int(val) == 1000 else "background-color: #f8d7da"

    styled = df_val.style.applymap(highlight, subset=data_cols)
    display(styled)

    facilities_done = len(df_val)
    all_pass = all(
        df_val[col].fillna(0).astype(int).eq(1000).all()
        for col in data_cols if col in df_val.columns
    )
    print(f"\nFacilities logged: {facilities_done}/10")
    print("GO" if all_pass and facilities_done == 10 else "NO-GO — still in progress or shortfalls present")

## Overall Completion Summary

In [ ]:
df_summary = q("""
    SELECT
        data_type,
        COUNT(*) FILTER (WHERE status = 'done')  AS done,
        COUNT(*) FILTER (WHERE status != 'done') AS remaining,
        COUNT(*)                                  AS total,
        ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'done') / NULLIF(COUNT(*), 0), 1) AS pct_done
    FROM public.pft_test_patient_work_queue
    WHERE execution_id = %s
    GROUP BY data_type
    ORDER BY data_type
""", (EXECUTION_ID,))

if df_summary.empty:
    print("No queue rows found.")
else:
    display(df_summary)
    total_done = df_summary["done"].sum()
    total_rows = df_summary["total"].sum()
    print(f"\nOverall: {total_done}/{total_rows} ({100*total_done/total_rows:.1f}%)")

## Recent NoETL Events (last 50)

Useful for spotting stuck steps, errors, or STATE-CACHE-STALE rebuilds.

In [ ]:
df_events = q("""
    SELECT
        event_id,
        event_type,
        step_name,
        status,
        created_at,
        metadata->>'commands_generated' AS cmds_generated,
        metadata->>'error'              AS error
    FROM noetl.event_log
    WHERE execution_id = %s
    ORDER BY event_id DESC
    LIMIT 50
""", (EXECUTION_ID,))
display(df_events)

## STATE-CACHE-STALE / LOOP-CACHE-RESTORE Events

Tracks how often the epoch-relative reset fix fires.

In [ ]:
df_stale = q("""
    SELECT
        event_id,
        event_type,
        step_name,
        created_at,
        metadata
    FROM noetl.event_log
    WHERE execution_id = %s
      AND (
          event_type ILIKE '%STATE-CACHE-STALE%'
          OR event_type ILIKE '%LOOP-CACHE-RESTORE%'
          OR (metadata::text ILIKE '%STATE-CACHE-STALE%')
          OR (metadata::text ILIKE '%LOOP-CACHE-RESTORE%')
      )
    ORDER BY event_id
""", (EXECUTION_ID,))

print(f"STATE-CACHE-STALE / LOOP-CACHE-RESTORE events: {len(df_stale)}")
display(df_stale)

## Loop Epoch Summary for `fetch_assessments`

Shows call.done counts per epoch key (each epoch = 1 pass of 100 patients).

In [ ]:
df_loops = q("""
    SELECT
        metadata->>'loop_key'   AS loop_key,
        metadata->>'step_name'  AS step_name,
        COUNT(*)                AS call_done_count,
        MIN(created_at)         AS first_event,
        MAX(created_at)         AS last_event
    FROM noetl.event_log
    WHERE execution_id = %s
      AND event_type = 'call.done'
      AND (step_name ILIKE '%fetch_assessments%'
           OR metadata->>'step_name' ILIKE '%fetch_assessments%')
    GROUP BY metadata->>'loop_key', metadata->>'step_name'
    ORDER BY MIN(created_at)
""", (EXECUTION_ID,))

if df_loops.empty:
    # Fallback: group by parent_step
    df_loops = q("""
        SELECT
            metadata->>'parent_step' AS parent_step,
            metadata->>'loop_key'    AS loop_key,
            COUNT(*)                 AS call_done_count,
            MIN(created_at)          AS first_event,
            MAX(created_at)          AS last_event
        FROM noetl.event_log
        WHERE execution_id = %s
          AND event_type = 'call.done'
        GROUP BY metadata->>'parent_step', metadata->>'loop_key'
        ORDER BY MIN(created_at)
    """, (EXECUTION_ID,))

print(f"Loop epoch groups: {len(df_loops)}")
display(df_loops)

## Step Completion Counts

In [ ]:
df_steps = q("""
    SELECT
        step_name,
        event_type,
        COUNT(*) AS count,
        MIN(created_at) AS first_seen,
        MAX(created_at) AS last_seen
    FROM noetl.event_log
    WHERE execution_id = %s
    GROUP BY step_name, event_type
    ORDER BY step_name, event_type
""", (EXECUTION_ID,))
display(df_steps)

## Close Connection

In [ ]:
conn.close()
print("Connection closed.")